# Kuhi anime-api Colab test

CPU 런타임용입니다. Google Drive는 사용하지 않습니다.
원본 Kuhi와 이 테스트 래퍼를 실행할 때마다 GitHub에서 clone합니다.

In [ ]:
!rm -rf /content/kuhi-anime-api /content/Data_crawling
!git clone -q https://github.com/aryaniiil/anime-api.git /content/kuhi-anime-api
!git clone -q https://github.com/HisameOgasahara/Data_crawling.git /content/Data_crawling
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -r /content/kuhi-anime-api/requirements.txt -r /content/Data_crawling/anime_api/requirements.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared


In [ ]:
import os
import subprocess
import time
import requests

env = os.environ.copy()
env["KUHI_ROOT"] = "/content/kuhi-anime-api"

api_process = subprocess.Popen(
    [
        "python",
        "-m",
        "uvicorn",
        "colab_app:app",
        "--app-dir",
        "/content/Data_crawling/anime_api",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/anime/providers/status",
            timeout=2,
        )
        if response.ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Kuhi API가 시작되지 않았습니다.")

print("Kuhi API ready: http://127.0.0.1:8000")


In [ ]:
QUERY = "Re:ZERO" # @param {type:"string"}
PER_PAGE = 12 # @param {type:"integer"}

import sys
from IPython.display import HTML, display

sys.path.insert(0, "/content/Data_crawling/anime_api")
from kuhi_client import KuhiClient

client = KuhiClient()
search_result = client.search(QUERY, per_page=PER_PAGE)
search_items = search_result.get("results", [])

cards = []
for index, item in enumerate(search_items):
    title = (
        (item.get("title") or {}).get("english")
        or (item.get("title") or {}).get("romaji")
        or str(item.get("id"))
    )
    cover = (item.get("coverImage") or {}).get("large", "")
    cards.append(
        f"""
        <div style="width:150px">
          <img src="{cover}" style="width:150px;height:225px;object-fit:cover;border-radius:8px">
          <div style="font-size:13px;margin-top:5px"><b>[{index}] {title}</b></div>
          <div style="font-size:12px">AniList ID: {item.get("id")}</div>
        </div>
        """
    )

display(
    HTML(
        '<div style="display:flex;gap:14px;flex-wrap:wrap">'
        + "".join(cards)
        + "</div>"
    )
)


In [ ]:
RESULT_INDEX = 0 # @param {type:"integer"}

from IPython.display import display
import pandas as pd

if RESULT_INDEX < 0 or RESULT_INDEX >= len(search_items):
    raise IndexError("RESULT_INDEX가 검색 결과 범위를 벗어났습니다.")

selected_item = search_items[RESULT_INDEX]
ANILIST_ID = str(selected_item["id"])
selected_title = (
    (selected_item.get("title") or {}).get("english")
    or (selected_item.get("title") or {}).get("romaji")
    or ANILIST_ID
)

print("선택:", selected_title)
print("AniList ID:", ANILIST_ID)

episodes_result = client.episodes(ANILIST_ID)
rows = []

for provider_name, provider_data in (episodes_result.get("providers") or {}).items():
    meta = provider_data.get("meta") or {}
    episodes = provider_data.get("episodes") or {}
    rows.append(
        {
            "provider": provider_name,
            "matched_title": meta.get("title"),
            "match_score": meta.get("matchScore"),
            "numbering": meta.get("numbering"),
            "sub_episodes": len(episodes.get("sub") or []),
            "dub_episodes": len(episodes.get("dub") or []),
            "raw_episodes": len(episodes.get("raw") or []),
        }
    )

display(pd.DataFrame(rows))


In [ ]:
EPISODE = 1 # @param {type:"integer"}
AUDIO = "sub" # @param ["sub", "dub"]
PROVIDER = "" # @param {type:"string"}

extract_result = client.extract(
    ANILIST_ID,
    episode=EPISODE,
    audio=AUDIO,
    provider=PROVIDER or None,
)
extract_result


In [ ]:
DOWNLOAD = True # @param {type:"boolean"}
STREAM_INDEX = 0 # @param {type:"integer"}
QUALITY = "best" # @param ["best", "1080", "720", "480", "worst"]
CONCURRENT_FRAGMENTS = 8 # @param [1, 4, 8, 16]
SUBTITLE_ENGLISH = False # @param {type:"boolean"}
SUBTITLE_JAPANESE = True # @param {type:"boolean"}
OUTPUT_FILE = "/content/downloads/anime_episode.mp4" # @param {type:"string"}

if DOWNLOAD:
    saved = client.download(
        extract_result,
        output_path=OUTPUT_FILE,
        stream_index=STREAM_INDEX,
        quality=QUALITY,
        subtitle_english=SUBTITLE_ENGLISH,
        subtitle_japanese=SUBTITLE_JAPANESE,
        concurrent_fragments=CONCURRENT_FRAGMENTS,
    )
    print("saved:", saved)
else:
    print("DOWNLOAD=False")


In [ ]:
import re

tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_url = None
deadline = time.time() + 30

while time.time() < deadline:
    line = tunnel_process.stdout.readline()
    if not line:
        if tunnel_process.poll() is not None:
            break
        time.sleep(0.2)
        continue

    print(line, end="")
    match = re.search(
        r"https://[-a-z0-9]+\.trycloudflare\.com",
        line,
    )
    if match:
        tunnel_url = match.group(0)
        break

if not tunnel_url:
    raise RuntimeError(
        "Cloudflare Tunnel URL을 찾지 못했습니다. 위 cloudflared 로그를 확인하세요."
    )

print("Viewer:", tunnel_url + "/viewer")
print("API:", tunnel_url)
